# A3.3 — Beğeni Tahmini Baseline (Kaggle verisiyle deneme)

**Amaç:** `likes_baseline.py` içindeki üretim fonksiyonlarını gerçek veritabanı olmadan,
Kaggle "Instagram Analytics" veri setiyle test etmek.

**Not (raporda belirtilecek):** Bu bir Kaggle veri seti, InstaScope'un kendi
veritabanından DEĞİL. Sonuçlar, modelin/pipeline'ın mantığının çalıştığını
doğrulamak içindir.

**Sütun eşleştirmesi:**
| Bizim özelliğimiz | CSV karşılığı |
|---|---|
| account_id | account_id |
| content_type | media_type |
| hour | post_hour |
| day_of_week | day_of_week (metin → ISO 1-7'ye çevrildi) |
| hashtag_count | hashtags_count |
| caption_length | caption_length |
| likes (hedef) | likes |
| account_growth_rate | follower_count'tan hesap başına türetildi |


## 1. Kurulum ve import

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src.ai.likes_baseline import (
    train_and_evaluate,
    FEATURE_COLUMNS_NUMERIC,
    FEATURE_COLUMNS_CATEGORICAL,
    TARGET_COLUMN,
)

pd.set_option("display.max_columns", 50)


## 2. CSV'yi örnekleyerek oku

Toplam ~30.000 satır var, hepsine gerek yok — hesap başına makul bir örneklem alıyoruz.

In [6]:
CSV_PATH = "../data/Instagram_Analytics.csv"  # kendi indirdiğin yere göre yolu güncelle
SAMPLE_PER_ACCOUNT = 500  # hesap başına kaç post alınacak 

raw = pd.read_csv(CSV_PATH)
print("toplam satır:", len(raw), "| hesap sayısı:", raw["account_id"].nunique())

sampled = pd.concat(
    [g.sample(n=min(len(g), SAMPLE_PER_ACCOUNT), random_state=42) for _, g in raw.groupby("account_id")]
).reset_index(drop=True)
print("örneklem sonrası satır:", len(sampled))
sampled.head()


toplam satır: 29999 | hesap sayısı: 20
örneklem sonrası satır: 10000


,post_id,account_id,account_type,follower_count,media_type,content_category,traffic_source,has_call_to_action,post_datetime,post_date,post_hour,day_of_week,likes,comments,shares,saves,reach,impressions,engagement_rate,followers_gained,caption_length,hashtags_count,performance_bucket_label
0,IG0018181,1,brand,9461,carousel,Beauty,Home Feed,1,2025-01-02 19:00:00,2025-01-02,19,Thursday,49,0,2,7,3110,3867,0.0150,811,131,7,low
1,IG0022413,1,brand,9461,image,Music,Profile,0,2025-08-05 03:00:00,2025-08-05,3,Tuesday,101,1,2,26,3280,3682,0.0353,95,145,8,medium
2,IG0008631,1,creator,9461,image,Food,External,0,2024-12-13 10:00:00,2024-12-13,10,Friday,560,19,21,76,9518,11340,0.0596,978,130,11,viral
3,IG0010438,1,creator,9461,reel,Food,Home Feed,0,2025-11-10 13:00:00,2025-11-10,13,Monday,317,6,17,40,4551,6151,0.0618,270,116,10,viral
4,IG0020840,1,creator,9461,carousel,Technology,External,0,2024-12-27 08:00:00,2024-12-27,8,Friday,75,2,0,10,5420,6016,0.0145,146,114,5,low


## 3. Özellik eşleştirme

`likes_baseline.py`'daki `build_feature_dataframe` metin/caption üzerinden hashtag/uzunluk hesaplıyordu — bu CSV'de zaten hazır geldiği için burada doğrudan eşliyoruz, üretim koduyla aynı SÜTUN ADLARINA getiriyoruz.

In [7]:
ISO_DAY_MAP = {
    "Monday": 1, "Tuesday": 2, "Wednesday": 3, "Thursday": 4,
    "Friday": 5, "Saturday": 6, "Sunday": 7,
}

df = sampled.copy()
df["posted_at"] = pd.to_datetime(df["post_datetime"])
df["hour"] = df["post_hour"]
df["day_of_week"] = df["day_of_week"].map(ISO_DAY_MAP)
df["hashtag_count"] = df["hashtags_count"]
df["caption_length"] = df["caption_length"]
df["content_type"] = df["media_type"]
df["likes"] = df["likes"]

# account_growth_rate: hesap başına follower_count'un zaman içindeki eğimi
# (üretimdeki account_metrics mantığının CSV karşılığı — ilk<->son ölçüm arası gün başına değişim)
def _growth_rate(group: pd.DataFrame) -> float:
    g = group.sort_values("posted_at")
    days = (g["posted_at"].iloc[-1] - g["posted_at"].iloc[0]).total_seconds() / 86400.0
    if days <= 0:
        return 0.0
    return (g["follower_count"].iloc[-1] - g["follower_count"].iloc[0]) / days

growth_by_account = {
    account_id: _growth_rate(group) for account_id, group in df.groupby("account_id")
}
df["account_growth_rate"] = df["account_id"].map(growth_by_account)

feature_df = df[
    ["account_id", "posted_at"] + FEATURE_COLUMNS_NUMERIC + FEATURE_COLUMNS_CATEGORICAL + [TARGET_COLUMN]
].copy()

feature_df.head()


,account_id,posted_at,hour,day_of_week,hashtag_count,caption_length,account_growth_rate,content_type,likes
0,1,2025-01-02 19:00:00,19,4,7,131,0.0,carousel,49
1,1,2025-08-05 03:00:00,3,2,8,145,0.0,image,101
2,1,2024-12-13 10:00:00,10,5,11,130,0.0,image,560
3,1,2025-11-10 13:00:00,13,1,10,116,0.0,reel,317
4,1,2024-12-27 08:00:00,8,5,5,114,0.0,carousel,75


## 4. Baseline modelleri eğit ve MAE karşılaştır

Burası `likes_baseline.py`'daki ÜRETIM `train_and_evaluate` fonksiyonu — hiç kopyalanmadı, doğrudan import edildi.

In [8]:
reports = []
for model_type in ("ridge", "gradient_boosting"):
    report = train_and_evaluate(feature_df, model_type=model_type)
    reports.append(report)
    status = "GEÇTİ ✅" if report.beats_naive else "GEÇEMEDİ ❌"
    print(
        f"[{report.model_type}] MAE={report.mae:.2f} | "
        f"naive_MAE={report.naive_mae:.2f} | "
        f"n_train={report.n_train} n_test={report.n_test} | DoD: {status}"
    )


[ridge] MAE=194.81 | naive_MAE=204.34 | n_train=7980 n_test=2000 | DoD: GEÇTİ ✅
[gradient_boosting] MAE=196.52 | naive_MAE=204.34 | n_train=7980 n_test=2000 | DoD: GEÇTİ ✅


## 5. Sonuç / Rapor notu

- Bu deneme **Kaggle** verisiyle yapıldı.
- `account_growth_rate`, CSV'deki `follower_count`'tan türetildi — gerçek projede
  `account_metrics` tablosundan aynı mantıkla hesaplanıyor (bkz. `likes_baseline.py`
  `fetch_account_growth()`), yani mantık tutarlı.
- Naif taban çizgisi ve train/test ayrımı üretim kodundaki KRONOLOJİK mantıkla birebir
  aynı (`shift(1)` ile leakage yok, hesap başına ilk %80 train / son %20 test).
- Gerçek DB doldukça bu notebook'u DB'den okuyan asıl `run_baseline_comparison()`
  fonksiyonuyla (main.py / likes_baseline.py `__main__` bloğu) tekrar çalıştırıp
  asıl MAE raporuna geçilecek.

## Sonuçlar

| Örneklem | Model | MAE | Naif MAE | DoD |
|---|---|---|---|---|
| 150/hesap (n_train=2.380) | Ridge | 197.54 | 203.67 | ✅ |
| 150/hesap (n_train=2.380) | Gradient Boosting | 204.85 | 203.67 | ❌ |
| 500/hesap (n_train=7.980) | Ridge | 194.81 | 204.34 | ✅ |
| 500/hesap (n_train=7.980) | Gradient Boosting | 196.52 | 204.34 | ✅ |

## Yorum

Küçük örneklemde (150/hesap) Gradient Boosting naif tahmini geçemedi —
yüksek model karmaşıklığı, az veriyle (2.380 satır) eğitim gürültüsünü
ezberledi (overfitting). Örneklem 500/hesaba çıkarılınca (n_train=7.980)
Gradient Boosting da DoD'u geçti, hipotez doğrulandı: veri arttıkça model
gürültü yerine gerçek örüntüyü öğrenmeye başladı. Ridge'in iyileşmesi
görece küçüktü (197.54 → 194.81) — düşük karmaşıklıklı modeller zaten az
veriyle de makul genelleme yapabiliyor, veri arttıkça kazanç azalıyor.

## Sonuç ve Öneri

Küçük/orta veri boyutunda **Ridge** daha güvenilir bir başlangıç noktası.
Bu nedenle üretimde başlangıçta Ridge kullanılması, InstaScope'un gerçek
veritabanı büyüdükçe Gradient Boosting'e geçişin yeniden değerlendirilmesi
önerilir.


## Deney: real_data_pool.json ile gerçek veride test 

Şu ana kadarki denemeler mock/Kaggle veriyle yapılmıştı. Burada `collect_pool.py` ile toplanan GERÇEK Instagram verisi kullanılıyor — DB'ye hiç bağlanmadan, `real_data_pool.json`'dan doğrudan post tablosu kuruluyor, sonra `likes_baseline.py`'daki GERÇEK fonksiyonlar (kod kopyalanmadı, doğrudan import edildi) çağrılıyor.

**Bilinen sınırlılık:** `account_growth_rate` özelliği normalde `account_metrics` tablosundaki gerçek takipçi geçmişinden hesaplanıyor — gerçek hesaplar için bu veri toplanmadığından hepsi `0.0` kullanılıyor. Ridge/GBM bu özelliğe zaten az ağırlık veriyordu, sonucu büyük ölçüde etkilemiyor ama raporda not edilmeli.

In [1]:
import os
import sys
sys.path.append(os.path.abspath(".."))

import json
import pandas as pd
from src.ai.likes_baseline import build_feature_dataframe, train_and_evaluate

with open("../data/real_data_pool.json", encoding="utf-8") as f:
    real_data = json.load(f)

rows = []
for account in real_data["accounts"]:
    username = account["profile"]["username"]
    for post in account["posts"]:
        rows.append({
            "account_id": username,
            "type": post["type"],
            "caption": post["caption"],
            "posted_at": post["created_at"],
            "likes": post["likes"],
        })

post_df = pd.DataFrame(rows)
post_df["posted_at"] = pd.to_datetime(post_df["posted_at"])

print(f"Toplam post: {len(post_df)} | hesap sayısı: {post_df['account_id'].nunique()}")
print(post_df.groupby("account_id").size())
print()
print(post_df.groupby("account_id")["likes"].describe()[["mean", "min", "max"]])


Toplam post: 71 | hesap sayısı: 8
account_id
ankaradaneyenir        15
ervapeh                15
havvahf                 5
kardelen.yildirim       4
nefisyemektarifleri     5
netflixturkiye          6
rumfitt_               15
sercankahvci            6
dtype: int64

                             mean      min       max
account_id                                          
ankaradaneyenir        713.200000     16.0    6360.0
ervapeh               7424.733333   1290.0   15335.0
havvahf              37223.400000  23633.0   69945.0
kardelen.yildirim    50267.750000   3760.0  148311.0
nefisyemektarifleri  14143.800000   7024.0   20798.0
netflixturkiye        9142.166667    437.0   20537.0
rumfitt_             23687.266667    454.0  239888.0
sercankahvci            70.500000     58.0      82.0


In [2]:
# account_growth_rate: gerçek follower geçmişi (account_metrics) elimizde
# yok, bilinen bir sınırlılık olarak 0.0 kullanılıyor (bkz. yukarıdaki not)
account_growth = {}

feature_df = build_feature_dataframe(post_df, account_growth)

for model_type in ["ridge", "gradient_boosting"]:
    try:
        report = train_and_evaluate(feature_df, model_type=model_type)
        status = "GEÇTİ ✅" if report.beats_naive else "GEÇEMEDİ ❌"
        print(
            f"[{model_type}] MAE={report.mae:.2f} | naive_MAE={report.naive_mae:.2f} "
            f"| n_train={report.n_train} n_test={report.n_test} | DoD: {status}"
        )
    except Exception as e:
        print(f"[{model_type}] HATA: {e}")


[ridge] MAE=8436.68 | naive_MAE=8082.23 | n_train=49 n_test=14 | DoD: GEÇEMEDİ ❌
[gradient_boosting] MAE=10356.85 | naive_MAE=8082.23 | n_train=49 n_test=14 | DoD: GEÇEMEDİ ❌


### Not — ölçek farkı

Hesaplar arası beğeni ölçeği çok farklı (ör. `sercankahvci` ortalama ~70, `rumfitt_` ortalama ~23.000, en yüksek tekil post 239.888 beğeni) — bu yüzden MAE mutlak olarak büyük görünebilir. Karşılaştırma yine de adil: naif taban çizgisi de AYNI aykırı değerlerden etkileniyor, `model < naive` kıyası bozulmuyor, sadece mutlak sayılar büyük çıkıyor.